# Change-impact explorer

Search any element and inspect its owner, children, and directly related elements.

This sample uses only standard SysML v2 concepts and automatically discovers the project's `model/` or `src/` directory.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import syside

def find_sysml_root(start=Path.cwd()):
    """Find the nearest model/ or src/ folder containing textual SysML."""
    for directory in (start, *start.parents):
        for folder_name in ('model', 'src'):
            candidate = directory / folder_name
            if candidate.is_dir() and next(candidate.rglob('*.sysml'), None):
                return candidate
    raise FileNotFoundError('No model/ or src/ directory containing .sysml files was found')

SYSML_ROOT = find_sysml_root()
SYSML_FILES = sorted(SYSML_ROOT.rglob('*.sysml'))
model, diagnostics = syside.try_load_model([str(path) for path in SYSML_FILES])
print(f'Loaded {len(SYSML_FILES)} SysML files from {SYSML_ROOT}')

In [ ]:
QUERY = ''  # Example: 'pump', 'controller', 'safety', or a requirement name

semantic_elements = (
    list(model.elements(syside.Usage, include_subtypes=True))
    + list(model.elements(syside.Definition, include_subtypes=True))
)

def label(element):
    value = element.qualified_name or element.name or element.declared_name
    return str(value) if value else '<unnamed>'

if not QUERY:
    print('Set QUERY to a name fragment. Suggestions:')
    for suggestion in [label(item) for item in semantic_elements if item.name][:20]:
        print(' -', suggestion)
    matches = []
else:
    matches = [item for item in semantic_elements if QUERY.casefold() in label(item).casefold()]
    print(f'{len(matches)} match(es) for {QUERY!r}')

[(type(item).__name__, label(item)) for item in matches[:100]]

In [ ]:
def neighborhood(element):
    owner = getattr(element, 'owner', None)
    children = [child for child in element.owned_elements if isinstance(child, (syside.Usage, syside.Definition))]
    related = []
    for relationship in element.owned_relationships:
        for candidate in relationship.related_elements:
            if candidate is not element:
                related.append(candidate)
    return {
        'selected': (type(element).__name__, label(element)),
        'owner': None if owner is None else (type(owner).__name__, label(owner)),
        'children': [(type(child).__name__, label(child)) for child in children[:50]],
        'directly related': [(type(item).__name__, label(item)) for item in related[:50]],
    }

[neighborhood(item) for item in matches[:10]]